Import `pandas` for data analysis and manipulation.

In [ ]:
import polars as pl

Read our scraped metadata CSV file into a DataFrame.

In [ ]:
df = pl.read_csv("../datasets/spotify_datasets_metadata.csv", try_parse_dates=True)
rows, cols = df.shape
print(f"DataFrame has {rows} rows and {cols} columns.")
print("Sample data:")
df.head()

For this project, we care about how a track characteristics affects its popularity. As track popularity is time-sensitive and can change over time, our approach will be to aggregate the data by year based on the `last_updated` column.

We will select the year that has the most data points to ensure a robust analysis.

In [ ]:
df.group_by(pl.col("last_updated").dt.year()).len().sort("last_updated")

We can see that from 2022 and onwards, there is a significant increase in the number of data points available. We now will focus on those.

In [ ]:
df = df.filter(pl.col("last_updated").dt.year() >= 2022)

Let's us start to do some filtering:
- include "lyric" in `title` or `subtitle`: likely to be lyrics datasets, and not actual song metadata.
- that has `download_count` in top 20 percentile: likely to be popular datasets.

In [ ]:
lyrics_df = df.filter(
    pl.col("title").str.contains("(?i)lyric")
    | pl.col("subtitle").str.contains("(?i)lyric")
)

In [ ]:
lyrics_df

We will keep those datasets as auxiliary datasets for future use.

In [ ]:
lyrics_df.write_csv("../datasets/spotify_lyrics_datasets_metadata.csv")

In [ ]:
df = df.filter(
    (pl.col("download_count") >= df["download_count"].quantile(0.8))
    & ~pl.col("id").is_in(lyrics_df["id"])
)

In [ ]:
df.group_by(pl.col("last_updated").dt.year()).len().sort("last_updated")

After filtering, we see that 2023 and 2024 has the most data points. Let's throughly check datasets in those years.

In [ ]:
df.filter(pl.col("last_updated").dt.year() == 2023)

After checking those, we can remove below datasets from our main analysis:
- `salvatorerastelli/spotify-and-youtube`: Does not contains track popularity.
- `rakkesharv/spotify-top-10000-streamed-songs`: Does not contains track popularity.
- `amaanansari09/top-100-songs`: only 100 samples, too small.
- `meeraajayakumar/spotify-user-behavior-dataset`: contains user behavior, not track metadata.


We further divide the datasets into two types:
- Main datasets: contains track popularity and audio features.
- Axuliary datasets: does not contains track popularity, but has other features that could use to enrich the main one.

In [ ]:
df_2023 = df.filter(
    (pl.col("last_updated").dt.year() == 2023)
    & ~pl.col("ref").is_in(
        [
            "salvatorerastelli/spotify-and-youtube",
            "rakkesharv/spotify-top-10000-streamed-songs",
            "amaanansari09/top-100-songs",
            "meeraajayakumar/spotify-user-behavior-dataset",
        ]
    )
)

In [ ]:
aux_2023_df = df_2023.filter(
    pl.col("ref").is_in(
        [
            "nelgiriyewithana/top-spotify-songs-2023",
            "undefinenull/million-song-dataset-spotify-lastfm",
        ]
    )
)
aux_2023_df

In [ ]:
aux_2023_df.write_csv("../datasets/spotify_aux_datasets_metadata.csv")

In [ ]:
# main_2023_df = df_2023[~df_2023.ref.isin(aux_2023_df.ref)]
main_2023_df = df_2023.filter(
    ~pl.col("ref").is_in(aux_2023_df["ref"])
)
main_2023_df

In [ ]:
main_2023_df.write_csv("../datasets/spotify_main_datasets_metadata.csv")

We use the same principle to examine the datasets in year 2024.

In [ ]:
df.filter(pl.col("last_updated").dt.year() == 2024)

Out of 9 datasets in 2024, we found that:
- `jarredpriester/taylor-swift-spotify-dataset`: only Taylor Swift songs, not representative.


For the rest 8 datasets, 5 of them do not contains track popularity, we will keep them as auxiliary datasets:
- `nelgiriyewithana/most-streamed-spotify-songs-2024`
- `abdulszz/spotify-most-streamed-songs`
- `arnavvvvv/spotify-music`
- `joebeachcapital/top-10000-spotify-songs-1960-now`
- `datasets/zeesolver/spotfy`


Only 3 remained datasets contain track popularity. But in this case, the total data points are too few (only `70728` unique `track_id` samples, compare to more than 1.5M in 2023 datasets). Therefore, we will not use any datasets from 2024 for our main analysis.

This conclude our data selection process.